In [25]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "True"



In [ ]:
from langsmith import Client

client = Client()

dataset_name = "simple chatbot evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id = dataset.id,
    examples=[
        {
            "inputs": {
                "question": "What is Python?"
            },
            "outputs": {
                "answer": "Python is a programming language."
            }
        },
        {
            "inputs": {
                "question": "What is machine learning?"
            },
            "outputs": {
                "answer": "Machine learning allows computers to learn patterns from data."
            }
        },
        {
            "inputs": {
                "question": "What is an API?"
            },
            "outputs": {
                "answer": "An API allows software applications to communicate with each other."
            }
        },
        {
            "inputs": {
                "question": "What is a database?"
            },
            "outputs": {
                "answer": "A database stores and organizes data for retrieval and management."
            }
        },
        {
            "inputs": {
                "question": "What is RAG in AI?"
            },
            "outputs": {
                "answer": "RAG retrieves relevant information from external sources to help an LLM generate grounded answers."
            }
        }
    ]
)

In [ ]:

from langchain_groq import ChatGroq

groq_client = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0
)

eval_instructions = """
You are a professor specialized in grading students' answers.
Judge whether the predicted answer is factually equivalent
to the reference answer. Ignore minor wording differences.
Respond with only one word: correct or incorrect.
"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    user_content = f"""
    Question: {inputs['question']}

    Reference answer: {reference_outputs['answer']}

    Predicted answer: {outputs['answer']}

    Is the predicted answer correct?
    """

    response = groq_client.invoke([
        ("system", eval_instructions),
        ("human", user_content)
    ])

    verdict = response.content.strip().lower()
    score = verdict.startswith("correct") and not verdict.startswith("incorrect")

    return {
        "key": "correctness",
        "score": score,
        "comment": verdict
    }


In [29]:
## concision

def concision(outputs: dict, reference_outputs: dict) -> dict:
    score = len(outputs["answer"]) < 2 * len(reference_outputs["answer"])

    return {
        "key": "concision",
        "score": score,
        "comment": "Checks whether the answer is less than twice the reference length."
    }


In [30]:

from langchain_groq import ChatGroq

default_instruction = (
    "Respond to the user question in a short, concise manner "
    "(one short answer)."
)

def my_app(
    question: str,
    model: str = "groq:qwen/qwen3.8-27b",
    instruction: str = default_instruction
) -> str:

    # ChatGroq expects the provider's model ID, without "groq:"
    llm = groq_client

    response = llm.invoke([
        ("system", instruction),
        ("human", question)
    ])

    return response.content


In [31]:
def ls_target(inputs: dict) -> dict:
    answer = my_app(inputs["question"])
    return {"answer": answer}

In [32]:
expereiment = client.evaluate(
    ls_target,
    data = dataset_name,
    evaluators =[correctness,concision],
    experiment_prefix=" simple chatbot evaluation",
    
)

View the evaluation results for experiment: ' simple chatbot evaluation-0c5f0045' at:
https://smith.langchain.com/o/b98f7d27-4cdc-4ee1-b9c9-9cd63e5997d8/datasets/89864806-681a-4ee9-88ef-b06e92478a7c/compare?selectedSessions=5851aedd-8d74-402d-9ea1-9abb571192e1




5it [00:03,  1.30it/s]
